# Phân tích kết quả Connect Four

Notebook này đọc log trong thư mục `logs/` và tạo ra các bảng, hình cho Chương 4 của báo cáo.
Chạy từ trên xuống dưới. Cần có `pandas` và `matplotlib`.

- Bảng lưu ở `analysis/tables/`, hình lưu ở `analysis/figures/notebook/`.
- Dải **dev** dùng để thử và chọn. Dải **test** (EXP-009) là số cuối, chỉ chạy một lần.
- Mọi con số đều lấy từ log. Không có số nào gõ tay.

| Phần | Nội dung | Log |
|---|---|---|
| 1 | Giải đấu test, chế độ A (cùng thời gian mỗi nước) | EXP-009-A |
| 2 | Giải đấu test, chế độ B (cấu hình mặc định) | EXP-009-B |
| 2b | So sánh các agent bằng 5 hình (score, win/draw/loss, đi trước/sau, chi phí, phân bố thời gian) | EXP-009-A, EXP-009-B |
| 3 | Các ván RL thua (để phân tích lỗi) | EXP-009-A, EXP-009-B |
| 4 | Ablation 1: bật/tắt pruning của Alpha-Beta | EXP-003, -D4, -D5 |
| 5 | Ablation 2: số simulations của MCTS | EXP-004 |
| 6 | RQ1: Alpha-Beta và MCTS cùng thời gian mỗi nước | EXP-005, EXP-009-A |
| 7 | Đường cong học của RL | RL-002, RL-003, RL-004 |
| 8 | Chọn checkpoint RL (dev) | EXP-008 |
| 9 | Tóm tắt nhanh |  |

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

try:
    from IPython.display import display
except ImportError:
    display = print

# Tìm thư mục gốc của dự án (notebook nằm trong thư mục analysis)
ROOT = Path.cwd()
if ROOT.name == "analysis":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from arena.metrics import wilson, percentile

LOGS = ROOT / "logs"
HINH = ROOT / "analysis" / "figures" / "notebook"
BANG = ROOT / "analysis" / "tables"
HINH.mkdir(parents=True, exist_ok=True)
BANG.mkdir(parents=True, exist_ok=True)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

RL = "dqn_RL-004_last"   # checkpoint RL đã chốt


def dang_k(x, _=None):
    # Viết gọn số lớn trên trục: 250000 -> 250k, 1000 -> 1k
    if x >= 1000:
        return f"{x / 1000:g}k"
    return f"{x:g}"
print("Thư mục gốc:", ROOT)

## Hàm dùng chung

- `doc_games(exp)`: đọc mọi ván của một thí nghiệm.
- `doc_moves(exp)`: đọc mọi nước đi (có `time_ms`, `nodes`, `depth`).
- `bang_cap(exp)`: bảng win/draw/loss và score cho từng cặp, kèm khoảng Wilson 95%.
- `bang_thoi_gian(exp)`: thời gian mỗi nước (mean, p95) của từng agent.
- `ve_ma_tran(...)`: ma trận đối đầu.

Score = (win + 0.5 × draw) / số ván. Cột `distinct_games` là số ván có chuỗi nước đi khác nhau trong 100 ván.
Nếu số này nhỏ thì các ván lặp lại nhiều, và khoảng Wilson trông chắc hơn thực tế.

In [ ]:
def doc_games(exp):
    files = sorted((LOGS / exp).glob("*/*_games.csv"))
    return pd.concat([pd.read_csv(f) for f in files], ignore_index=True)


def doc_moves(exp):
    files = sorted((LOGS / exp).glob("*/*_moves.csv"))
    return pd.concat([pd.read_csv(f) for f in files], ignore_index=True)


def diem(d, a):
    # score của agent a trong các ván d
    if len(d) == 0:
        return np.nan
    return ((d["winner"] == a).sum() + 0.5 * (d["winner"] == "draw").sum()) / len(d)


def bang_cap(exp):
    g = doc_games(exp)
    m = doc_moves(exp)
    hang = []
    for pair, d in g.groupby("pair", sort=False):
        a = d["agent_a"].iloc[0]
        b = d["agent_b"].iloc[0]
        n = len(d)
        win = int((d["winner"] == a).sum())
        draw = int((d["winner"] == "draw").sum())
        loss = n - win - draw
        lo, hi = wilson(win + 0.5 * draw, n)
        # số ván khác nhau: ghép các cột của từng ván thành một chuỗi
        chuoi = m[m["pair"] == pair].groupby("game")["col"].apply(lambda s: ",".join(map(str, s)))
        hang.append(dict(
            A=a, B=b, n=n, win=win, draw=draw, loss=loss,
            score=diem(d, a), ci_low=lo, ci_high=hi,
            score_first=diem(d[d["a_first"] == 1], a),
            score_second=diem(d[d["a_first"] == 0], a),
            distinct_games=chuoi.nunique(), mean_moves=d["moves"].mean()))
    return pd.DataFrame(hang)


def lay_score(bang, a, b):
    # score của agent a khi đấu b (tìm cả hai chiều của cặp)
    r = bang[(bang["A"] == a) & (bang["B"] == b)]
    if len(r) > 0:
        return r.iloc[0]["score"]
    r = bang[(bang["A"] == b) & (bang["B"] == a)]
    return 1 - r.iloc[0]["score"]


def bang_thoi_gian(exp):
    m = doc_moves(exp)
    return m.groupby("agent").agg(
        moves=("time_ms", "count"),
        mean_ms=("time_ms", "mean"),
        p95_ms=("time_ms", lambda s: percentile(list(s), 0.95)),
        mean_nodes=("nodes", "mean"),
        mean_depth=("depth", "mean"))


def ve_ma_tran(bang, ten_file, tieu_de):
    # Ô (hàng, cột) = score của agent ở hàng khi đấu agent ở cột
    ten = sorted(set(bang["A"]) | set(bang["B"]))
    mt = pd.DataFrame(np.nan, index=ten, columns=ten)
    for _, r in bang.iterrows():
        mt.loc[r["A"], r["B"]] = r["score"]
        mt.loc[r["B"], r["A"]] = 1 - r["score"]
    thu_tu = mt.mean(axis=1).sort_values(ascending=False).index
    mt = mt.loc[thu_tu, thu_tu]
    fig, ax = plt.subplots(figsize=(6.5, 5.2))
    ax.imshow(mt.values, cmap="RdYlGn", vmin=0, vmax=1)
    ax.set_xticks(range(len(ten)))
    ax.set_yticks(range(len(ten)))
    ax.set_xticklabels(mt.columns, rotation=40, ha="right")
    ax.set_yticklabels(mt.index)
    for i in range(len(ten)):
        for j in range(len(ten)):
            if not np.isnan(mt.values[i, j]):
                ax.text(j, i, f"{mt.values[i, j]:.2f}", ha="center", va="center", fontsize=9)
    ax.set_title(tieu_de)
    fig.tight_layout()
    fig.savefig(HINH / ten_file, dpi=200)
    plt.show()
    return mt

## 1 và 2. Giải đấu test

Hai chế độ ngân sách:
- **A**: Alpha-Beta và MCTS cùng 50 ms mỗi nước (`ab-t50`, `mcts-t50`).
- **B**: cấu hình mặc định (`ab-d4`, `mcts-s1000`).

Mỗi cặp 100 ván, chia đều đi trước/đi sau, dải seed test.

In [ ]:
KQ = {}


def chay_giai_dau(exp, tieu_de):
    bang = bang_cap(exp)
    tg = bang_thoi_gian(exp)
    bang.round(3).to_csv(BANG / f"{exp}_cap.csv", index=False, encoding="utf-8-sig")
    tg.round(3).to_csv(BANG / f"{exp}_thoi_gian.csv", encoding="utf-8-sig")
    print(tieu_de, "- kết quả từng cặp (score là của agent A)")
    display(bang.round(3))
    mt = ve_ma_tran(bang, f"{exp}_ma_tran.png", tieu_de)
    print("Thời gian mỗi nước (ms):")
    display(tg.round(2))
    KQ[exp] = (bang, mt, tg)


chay_giai_dau("EXP-009-A", "Test, chế độ A (50 ms mỗi nước)")

In [ ]:
chay_giai_dau("EXP-009-B", "Test, chế độ B (cấu hình mặc định)")

In [ ]:
# Score tách theo đi trước / đi sau của agent RL, đối với từng đối thủ
hang = []
for exp, (bang, mt, tg) in KQ.items():
    g = doc_games(exp)
    for dt in mt.columns:
        if dt == RL:
            continue
        d = g[((g["agent_a"] == RL) & (g["agent_b"] == dt)) | ((g["agent_a"] == dt) & (g["agent_b"] == RL))]
        # RL đi trước khi (RL là A và a_first=1) hoặc (RL là B và a_first=0)
        rl_truoc = d[((d["agent_a"] == RL) & (d["a_first"] == 1)) | ((d["agent_b"] == RL) & (d["a_first"] == 0))]
        rl_sau = d.drop(rl_truoc.index)
        hang.append(dict(exp=exp, doi_thu=dt, rl_score=diem(d, RL),
                         rl_score_di_truoc=diem(rl_truoc, RL), rl_score_di_sau=diem(rl_sau, RL)))
bang_rl = pd.DataFrame(hang)
bang_rl.round(3).to_csv(BANG / "EXP-009_rl_truoc_sau.csv", index=False, encoding="utf-8-sig")
display(bang_rl.round(3))

## 2b. So sánh các agent bằng hình (test)

Năm hình cho hai chế độ A và B:
1. Score trung bình của từng agent.
2. Win/draw/loss của RL trước từng đối thủ.
3. Score khi đi trước và khi đi sau.
4. Sức mạnh so với chi phí (score và ms mỗi nước).
5. Phân bố thời gian mỗi nước.

Lưu ý: Random có thời gian gần 0 ms, nên được đặt thành 0.01 ms để vẽ trục log.

In [ ]:
CHE_DO = [("EXP-009-A", "Chế độ A (50 ms mỗi nước)"), ("EXP-009-B", "Chế độ B (mặc định)")]
MAU = {"win": "tab:blue", "draw": "lightgray", "loss": "tab:orange"}


def doc_thoi_gian_theo_agent(exp):
    # Gắn tên agent (spec) vào từng nước đi: người đi nước 1 của ván là A nếu a_first = 1
    m = doc_moves(exp)
    g = doc_games(exp)[["pair", "game", "agent_a", "agent_b", "a_first"]]
    m = m.merge(g, on=["pair", "game"])
    la_a = (m["player"] == 1) == (m["a_first"] == 1)
    m["spec"] = np.where(la_a, m["agent_a"], m["agent_b"])
    return m


def bang_truoc_sau(exp):
    # score trung bình của mỗi agent khi đi trước (True) và khi đi sau (False)
    g = doc_games(exp)
    hang = []
    for _, r in g.iterrows():
        sa = 1.0 if r["winner"] == r["agent_a"] else (0.5 if r["winner"] == "draw" else 0.0)
        hang.append((r["agent_a"], r["a_first"] == 1, sa))
        hang.append((r["agent_b"], r["a_first"] != 1, 1 - sa))
    d = pd.DataFrame(hang, columns=["agent", "di_truoc", "score"])
    return d.groupby(["agent", "di_truoc"])["score"].mean().unstack()


# Hình 1: score trung bình của từng agent
fig, ax = plt.subplots(1, 2, figsize=(9, 3.6), sharex=True)
for a, (exp, ten) in zip(ax, CHE_DO):
    tb = KQ[exp][1].mean(axis=1).sort_values()
    mau = ["tab:red" if n == RL else "tab:blue" for n in tb.index]
    a.barh(tb.index, tb.values, color=mau)
    for i, v in enumerate(tb.values):
        a.text(v + 0.01, i, f"{v:.2f}", va="center")
    a.set_title(ten)
    a.set_xlim(0, 1.1)
    a.set_xlabel("score trung bình (trên mọi đối thủ)")
fig.tight_layout()
fig.savefig(HINH / "so_sanh_1_score_trung_binh.png", dpi=200)
plt.show()

In [ ]:
# Hình 2: win/draw/loss của RL trước từng đối thủ (100 ván mỗi đối thủ)
fig, ax = plt.subplots(1, 2, figsize=(9, 3.2), sharex=True)
for a, (exp, ten) in zip(ax, CHE_DO):
    bang = KQ[exp][0]
    hang = []
    for _, r in bang.iterrows():
        if r["A"] == RL:
            hang.append((r["B"], r["win"], r["draw"], r["loss"]))
        elif r["B"] == RL:
            hang.append((r["A"], r["loss"], r["draw"], r["win"]))
    d = pd.DataFrame(hang, columns=["doi_thu", "win", "draw", "loss"]).set_index("doi_thu")
    d = d.loc[d["win"].sort_values().index]
    trai = np.zeros(len(d))
    for cot in ["win", "draw", "loss"]:
        a.barh(d.index, d[cot], left=trai, color=MAU[cot], label=cot)
        trai = trai + d[cot].values
    a.set_title("RL-004 last, " + ten)
    a.set_xlabel("số ván (RL là bên được tính win)")
ax[0].legend(loc="upper center", bbox_to_anchor=(1.1, -0.3), ncol=3, fontsize=9)
fig.tight_layout()
fig.savefig(HINH / "so_sanh_2_win_draw_loss_RL.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# Hình 3: score khi đi trước và khi đi sau của từng agent
fig, ax = plt.subplots(1, 2, figsize=(9, 3.8), sharey=True)
for a, (exp, ten) in zip(ax, CHE_DO):
    d = bang_truoc_sau(exp)
    d = d.loc[d.mean(axis=1).sort_values(ascending=False).index]
    x = np.arange(len(d))
    a.bar(x - 0.2, d[True], 0.4, label="đi trước", color="tab:blue")
    a.bar(x + 0.2, d[False], 0.4, label="đi sau", color="tab:orange")
    a.set_xticks(x)
    a.set_xticklabels(d.index, rotation=40, ha="right")
    a.set_title(ten)
    a.set_ylim(0, 1.05)
ax[0].set_ylabel("score")
ax[0].legend(fontsize=8)
fig.tight_layout()
fig.savefig(HINH / "so_sanh_3_di_truoc_di_sau.png", dpi=200)
plt.show()

In [ ]:
# Hình 4: sức mạnh so với chi phí (mỗi chấm là một agent)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.9), sharey=True)
for a, (exp, ten) in zip(ax, CHE_DO):
    m = doc_thoi_gian_theo_agent(exp)
    ms = m.groupby("spec")["time_ms"].mean().clip(lower=0.01)
    sc = KQ[exp][1].mean(axis=1)
    for ten_agent in sc.index:
        a.scatter(ms[ten_agent], sc[ten_agent], color="tab:red" if ten_agent == RL else "tab:blue")
        a.annotate(ten_agent, (ms[ten_agent], sc[ten_agent]), textcoords="offset points", xytext=(5, 5), fontsize=8)
    a.set_xscale("log")
    a.set_xlim(0.005, 300)
    a.set_title(ten)
    a.set_xlabel("ms mỗi nước (trung bình, trục log)")
ax[0].set_ylabel("score trung bình")
fig.tight_layout()
fig.savefig(HINH / "so_sanh_4_suc_manh_chi_phi.png", dpi=200)
plt.show()

In [ ]:
# Hình 5: phân bố thời gian mỗi nước (hộp: 25%-75%, vạch giữa là median, râu không tính điểm ngoài)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.9), sharey=True)
for a, (exp, ten) in zip(ax, CHE_DO):
    m = doc_thoi_gian_theo_agent(exp)
    thu_tu = m.groupby("spec")["time_ms"].median().sort_values().index
    data = [np.clip(m.loc[m["spec"] == s, "time_ms"].values, 0.01, None) for s in thu_tu]
    a.boxplot(data, tick_labels=list(thu_tu), showfliers=False)
    a.set_yscale("log")
    a.set_title(ten)
    a.tick_params(axis="x", rotation=40)
ax[0].set_ylabel("ms mỗi nước (trục log)")
fig.tight_layout()
fig.savefig(HINH / "so_sanh_5_phan_bo_thoi_gian.png", dpi=200)
plt.show()

## 3. Các ván RL thua

Dùng để tìm tình huống RL mắc lỗi. Khi chọn ván để phân tích, hãy dùng một quy tắc cố định
(ví dụ lấy 3 ván có số `game` nhỏ nhất của mỗi cặp), không chọn riêng ván đẹp.

In [ ]:
for exp in ["EXP-009-A", "EXP-009-B"]:
    g = doc_games(exp)
    thua = g[((g["agent_a"] == RL) & (g["winner"] == g["agent_b"])) |
             ((g["agent_b"] == RL) & (g["winner"] == g["agent_a"]))]
    print(exp, "- số ván RL thua:", len(thua))
    if len(thua) > 0:
        display(thua[["pair", "game", "a_first", "winner", "moves"]].reset_index(drop=True))
    hoa = g[(g["winner"] == "draw") & ((g["agent_a"] == RL) | (g["agent_b"] == RL))]
    print(exp, "- số ván RL hòa:", len(hoa))

### 3b. Cách thua và ván điển hình (test)

- **forced_loss**: ở nước cuối của bên thua, không còn nước nào tránh được thua ngay (thua cưỡng bức).
- **one_move_mistake**: còn nước an toàn nhưng bên thua không chọn.
- Chỉ xét MỘT nước cuối của bên thua. Sai lầm xảy ra sớm hơn (đã đẩy ván vào thế thua cưỡng bức) không bị phát hiện.
- Ván điển hình chọn theo quy tắc cố định: 3 ván RL thua có số `game` nhỏ nhất của mỗi chế độ.

In [ ]:
from analysis.report_analysis import classify_loss, replay, winning_cells, draw_board


def doc_cols(exp):
    # chuỗi cột của từng ván: {(pair, game): [cột, ...]}
    m = doc_moves(exp).sort_values(["pair", "game", "ply"])
    return {k: [int(c) for c in v["col"]] for k, v in m.groupby(["pair", "game"])}


def cach_thua(exp):
    g = doc_games(exp)
    cols = doc_cols(exp)
    dem = {}
    for _, r in g.iterrows():
        if r["winner"] == "draw":
            continue
        thua = r["agent_b"] if r["winner"] == r["agent_a"] else r["agent_a"]
        nhom, loai = classify_loss(cols[(r["pair"], r["game"])])
        d = dem.setdefault(thua, {"loss": 0, "forced_loss": 0, "one_move_mistake": 0})
        d["loss"] += 1
        d["forced_loss" if nhom == "cưỡng bức" else "one_move_mistake"] += 1
    return pd.DataFrame(dem).T.sort_values("loss", ascending=False)


for exp, ten in CHE_DO:
    t = cach_thua(exp)
    t.to_csv(BANG / f"{exp}_cach_thua.csv", encoding="utf-8-sig")
    print(ten)
    display(t)

In [ ]:
# Ván điển hình: 3 ván RL thua có số game nhỏ nhất của mỗi chế độ
fig, axes = plt.subplots(2, 3, figsize=(8, 6))
chon_van = []
for hang, (exp, ten) in enumerate(CHE_DO):
    g = doc_games(exp)
    cols = doc_cols(exp)
    thua = g[((g["agent_a"] == RL) & (g["winner"] == g["agent_b"])) | ((g["agent_b"] == RL) & (g["winner"] == g["agent_a"]))]
    thua = thua.sort_values(["game", "pair"]).head(3)
    for cot, (_, r) in enumerate(thua.iterrows()):
        day = cols[(r["pair"], r["game"])]
        b = replay(day)
        last = None
        if b.history:
            c, rr = b.history[-1]
            last = (rr, c)
        di_truoc = r["agent_a"] if r["a_first"] == 1 else r["agent_b"]
        draw_board(axes[hang][cot], b.grid, winning_cells(b), last,
                   f"{exp[-1]}: {r['pair'].replace('__vs__', ' vs ')}\nván {r['game']}, {len(day)} nước\n{di_truoc} đi trước", fs=7.5)
        chon_van.append(dict(exp=exp, pair=r["pair"], game=int(r["game"]), a_first=int(r["a_first"]), plies=len(day),
                             moves_1_to_7=" ".join(str(c + 1) for c in day)))
fig.tight_layout()
fig.savefig(HINH / "van_RL_thua.png", dpi=200)
plt.show()
van_chon = pd.DataFrame(chon_van)
van_chon.to_csv(BANG / "EXP-009_van_RL_thua_chon.csv", index=False, encoding="utf-8-sig")
display(van_chon)

## 4. Ablation 1: bật/tắt pruning của Alpha-Beta (dev)

So sánh `ab-dN` (có pruning) và `ab-dN-nopr` (minimax thuần) ở depth 3, 4, 5. Cột `ratio` = tắt / bật.
Ở cuối ô có thêm score theo đi trước/đi sau: nếu hai bên chọn cùng nước đi thì kết quả chỉ phụ thuộc người đi trước.

In [ ]:
hang = []
for exp in ["EXP-003", "EXP-003-D4", "EXP-003-D5"]:
    m = doc_moves(exp)
    t = m.groupby("agent").agg(nodes=("nodes", "mean"), ms=("time_ms", "mean"),
                               p95=("time_ms", lambda s: percentile(list(s), 0.95)))
    ten = [a for a in t.index if not a.endswith("-nopr")][0]
    bat, tat = t.loc[ten], t.loc[ten + "-nopr"]
    hang.append(dict(
        depth=int(ten.split("-d")[1]),
        nodes_pruning=bat["nodes"], nodes_nopr=tat["nodes"], nodes_ratio=tat["nodes"] / bat["nodes"],
        ms_pruning=bat["ms"], ms_nopr=tat["ms"], ms_ratio=tat["ms"] / bat["ms"],
        p95_pruning=bat["p95"], p95_nopr=tat["p95"]))
bang_cattia = pd.DataFrame(hang)
bang_cattia.round(2).to_csv(BANG / "EXP-003_pruning.csv", index=False, encoding="utf-8-sig")
display(bang_cattia.round(2))

fig, ax = plt.subplots(1, 2, figsize=(8, 3.4))
x = np.arange(len(bang_cattia))
for i, (cot, nhan) in enumerate([("nodes", "nodes mỗi nước"), ("ms", "ms mỗi nước")]):
    ax[i].bar(x - 0.2, bang_cattia[cot + "_pruning"], 0.4, label="có pruning")
    ax[i].bar(x + 0.2, bang_cattia[cot + "_nopr"], 0.4, label="tắt pruning")
    ax[i].set_xticks(x)
    ax[i].set_xticklabels([f"depth {d}" for d in bang_cattia["depth"]])
    ax[i].set_yscale("log")
    ax[i].set_ylabel(nhan)
ax[0].legend()
fig.tight_layout()
fig.savefig(HINH / "EXP-003_pruning.png", dpi=200)
plt.show()

print("Score của ab-dN (có pruning) khi đấu bản tắt pruning:")
for exp in ["EXP-003", "EXP-003-D4", "EXP-003-D5"]:
    display(bang_cap(exp)[["A", "B", "win", "draw", "loss", "score", "score_first", "score_second", "distinct_games"]].round(3))

## 5. Ablation 2: số simulations của MCTS (dev)

MCTS với 200, 1000, 5000 simulations, so với nhau và so với `ab-d4`.

In [ ]:
b4 = bang_cap("EXP-004")
tg4 = bang_thoi_gian("EXP-004")
b4.round(3).to_csv(BANG / "EXP-004_cap.csv", index=False, encoding="utf-8-sig")
display(b4.round(3))

sims = [200, 1000, 5000]
diem_ab = [lay_score(b4, f"mcts-s{s}", "ab-d4") for s in sims]
ms = [tg4.loc[f"MCTS-s{s}", "mean_ms"] for s in sims]
bang_sims = pd.DataFrame(dict(simulations=sims, score_vs_ab_d4=diem_ab, ms_per_move=ms))
bang_sims.round(3).to_csv(BANG / "EXP-004_simulations.csv", index=False, encoding="utf-8-sig")
display(bang_sims.round(3))

fig, ax = plt.subplots(1, 2, figsize=(8, 3.4))
ax[0].plot(sims, diem_ab, "o-")
ax[0].axhline(0.5, color="gray", ls="--", lw=1)
ax[0].set_xscale("log")
ax[0].set_xlabel("simulations")
ax[0].set_ylabel("score khi đấu ab-d4")
ax[1].plot(sims, ms, "o-", color="tab:orange")
ax[1].set_xscale("log")
ax[1].set_yscale("log")
ax[1].set_xlabel("simulations")
ax[1].set_ylabel("ms mỗi nước")
for a in ax:
    a.set_xticks(sims)
    a.set_xticklabels([dang_k(s) for s in sims])
    a.minorticks_off()
fig.tight_layout()
fig.savefig(HINH / "EXP-004_simulations.png", dpi=200)
plt.show()

## 6. RQ1: Alpha-Beta và MCTS cùng thời gian mỗi nước

Alpha-Beta (`ab-t`) và MCTS (`mcts-t`) có cùng giới hạn thời gian. Score là của Alpha-Beta.
Dev có hai mức (50 ms và 200 ms). Test chỉ có mức 50 ms.

In [ ]:
muc = [("dev 50 ms", "EXP-005-T50", "ab-t50", "mcts-t50"),
       ("dev 200 ms", "EXP-005-T200", "ab-t200", "mcts-t200"),
       ("test 50 ms", "EXP-009-A", "ab-t50", "mcts-t50")]
hang = []
for ten, exp, a, b in muc:
    bang = bang_cap(exp)
    r = bang[(bang["A"] == a) & (bang["B"] == b)].iloc[0]
    hang.append(dict(muc=ten, win=r["win"], draw=r["draw"], loss=r["loss"], score_ab=r["score"],
                     ci_low=r["ci_low"], ci_high=r["ci_high"], score_first=r["score_first"],
                     score_second=r["score_second"], distinct_games=r["distinct_games"]))
rq1 = pd.DataFrame(hang)
rq1.round(3).to_csv(BANG / "RQ1.csv", index=False, encoding="utf-8-sig")
display(rq1.round(3))

# Alpha-Beta đạt depth bao nhiêu, MCTS chạy bao nhiêu simulations trong cùng thời gian
for exp in ["EXP-005-T50", "EXP-005-T200"]:
    print(exp)
    display(bang_thoi_gian(exp)[["mean_ms", "p95_ms", "mean_nodes", "mean_depth"]].round(2))

fig, ax = plt.subplots(figsize=(5, 3.4))
yerr = [rq1["score_ab"] - rq1["ci_low"], rq1["ci_high"] - rq1["score_ab"]]
ax.bar(rq1["muc"], rq1["score_ab"], yerr=yerr, capsize=4, color="tab:blue")
ax.axhline(0.5, color="gray", ls="--", lw=1)
ax.set_ylim(0, 1)
ax.set_ylabel("score của Alpha-Beta")
fig.tight_layout()
fig.savefig(HINH / "RQ1.png", dpi=200)
plt.show()

## 7. Đường cong học của RL

Điểm dev trước Random và Heuristic, đo mỗi 2.500 ván.
- RL-002 học từ đầu (dừng ở 157.296 ván).
- RL-003 khởi động từ trọng số RL-002, nên trục x của RL-003 bắt đầu lại từ 0.
- RL-004 học từ đầu, cùng cấu hình RL-002, khác seed.

In [ ]:
runs = ["RL-002", "RL-003", "RL-004"]
fig, ax = plt.subplots(1, 2, figsize=(9, 3.4), sharey=True)
hang = []
for run in runs:
    t = pd.read_csv(LOGS / run / "train_log.csv")
    ax[0].plot(t["episode"], t["dev_random_score"], label=run)
    ax[1].plot(t["episode"], t["dev_heuristic_score"], label=run)
    hang.append(dict(run=run, episodes=int(t["episode"].iloc[-1]), train_seconds=t["elapsed_s"].iloc[-1],
                     dev_random_cuoi=t["dev_random_score"].iloc[-1], dev_heuristic_cuoi=t["dev_heuristic_score"].iloc[-1]))
ax[0].set_title("dev: đấu Random")
ax[1].set_title("dev: đấu Heuristic")
for a in ax:
    a.set_xlabel("episode")
    a.xaxis.set_major_formatter(FuncFormatter(dang_k))
    a.set_ylim(0, 1.05)
ax[0].set_ylabel("score")
ax[0].legend()
fig.tight_layout()
fig.savefig(HINH / "RL_learning_curve.png", dpi=200)
plt.show()
tom_tat_rl = pd.DataFrame(hang)
tom_tat_rl.round(3).to_csv(BANG / "RL_tom_tat.csv", index=False, encoding="utf-8-sig")
display(tom_tat_rl.round(3))

## 8. Chọn checkpoint RL (dev, EXP-008)

Quy tắc đặt trước khi có kết quả RL-004:
1. Ứng viên: `dqn_RL-003_last` và các checkpoint RL-004 từ `ep100000` đến `last`. Không dùng `best.pt`.
2. Tiêu chí: trung bình score dev trước `ab-d3`, `ab-d4`, `mcts-s1000`.
3. Nếu hai ứng viên chênh nhau không quá 0.03 thì chọn checkpoint có nhiều ván huấn luyện hơn.

In [ ]:
b8 = bang_cap("EXP-008")
ung_vien = ["dqn_RL-003_last"] + [f"dqn_RL-004_{c}" for c in ["ep100000", "ep120000", "ep140000", "ep160000", "ep180000", "last"]]
doi_thu = ["ab-d3", "ab-d4", "mcts-s1000"]
hang = []
for c in ung_vien:
    d = {dt: lay_score(b8, c, dt) for dt in doi_thu}
    d["trung_binh"] = np.mean(list(d.values()))
    hang.append(pd.Series(d, name=c))
chon = pd.DataFrame(hang).sort_values("trung_binh", ascending=False)
chon.round(3).to_csv(BANG / "EXP-008_chon_checkpoint.csv", encoding="utf-8-sig")
display(chon.round(3))
top = chon["trung_binh"].iloc[:2]
print("Cao nhất:", chon.index[0], "| chênh với hạng 2:", round(top.iloc[0] - top.iloc[1], 3))

## 9. Tóm tắt nhanh

Ô dưới in các con số chính từ log, để đối chiếu khi viết báo cáo.

In [ ]:
for exp, (bang, mt, tg) in KQ.items():
    tb = mt.mean(axis=1).sort_values(ascending=False)
    print(exp, "- score trung bình của mỗi agent (cao xuống thấp):")
    for ten, v in tb.items():
        print(f"   {ten}: {v:.3f}")
    print(f"   Agent nhanh nhất mỗi nước: {tg['mean_ms'].idxmin()} ({tg['mean_ms'].min():.2f} ms)")
    print(f"   Agent chậm nhất mỗi nước: {tg['mean_ms'].idxmax()} ({tg['mean_ms'].max():.2f} ms)")
    print()

print("Pruning (tắt / bật):")
for _, r in bang_cattia.iterrows():
    print(f"   depth {int(r['depth'])}: nodes gấp {r['nodes_ratio']:.1f} lần, thời gian gấp {r['ms_ratio']:.1f} lần")
print()
print("RQ1, score của Alpha-Beta:")
for _, r in rq1.iterrows():
    print(f"   {r['muc']}: {r['score_ab']:.3f} (Wilson {r['ci_low']:.2f}-{r['ci_high']:.2f})")

### Cần nhớ khi viết nhận xét

- **Dev và test khác nhau.** Số ở dev dùng để chọn. Chỉ số ở test (EXP-009) mới là kết quả cuối.
- **Ván lặp lại.** RL và Alpha-Beta thường chơi giống nhau giữa các ván, nên cột `distinct_games` có thể nhỏ.
  Khi đó khoảng Wilson hẹp hơn độ tin cậy thật.
- **Điểm theo đi trước / đi sau.** Nếu hai cột này chênh nhau nhiều, hãy nói rõ thế đi trước ảnh hưởng thế nào.
- **Sức mạnh thực nghiệm khác tối ưu.** Kết quả chỉ đúng cho đúng các cấu hình đã chạy (depth, số simulations, thời gian).
- **Chưa đo RAM/VRAM.** Log arena không ghi bộ nhớ. Ghi rõ điều này trong báo cáo nếu không đo thêm.
- **Chưa kiểm tra riêng ảnh hưởng của seed huấn luyện.** RL-002 và RL-004 cùng cấu hình khác seed, nhưng chỉ có hai seed.